# 阶段2验证：表达式文法与搜索空间

本 Notebook 不读取行情数据。它检查142个 Token、规范化部分 AST、开放槽位联合动作、固定均匀后向概率、随机合法 DAG 轨迹和结构去重。所有检查均通过后，最后一个单元格会输出通过摘要。

In [ ]:
from pathlib import Path
import sys
import numpy as np

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from factor_gfn.grammar import (
    BINARY_OPERATORS, CROSS_SECTIONAL_OPERATORS, LEAVES,
    TS_BINARY_OPERATORS, TS_UNARY_OPERATORS, UNARY_OPERATORS,
    TOTAL_ACTIONS, WINDOWS, DAGAction, Expression, GrammarState,
    action_space_fingerprint, action_space_manifest,
    state_space_fingerprint, transition_space_fingerprint,
    get_action_id, validate_postfix,
)

print('project_root:', project_root)
print('Python:', sys.version.split()[0])

In [ ]:
EXPECTED_FINGERPRINT = '5689dbceb1bb42716773bcaf4cb5845041e578a3bb11fe67445ede6cde7938cc'

assert len(LEAVES) == 6
assert len(UNARY_OPERATORS) == 12
assert len(TS_UNARY_OPERATORS) == 17
assert len(BINARY_OPERATORS) == 10
assert len(TS_BINARY_OPERATORS) == 4
assert len(CROSS_SECTIONAL_OPERATORS) == 9
assert WINDOWS == (5, 10, 20, 40, 60)
assert TOTAL_ACTIONS == 142
assert len(action_space_manifest()) == TOTAL_ACTIONS
assert action_space_fingerprint() == EXPECTED_FINGERPRINT

initial = GrammarState(max_depth=4, max_nodes=12)
slot = initial.open_slots()[0]
mask = initial.get_legal_token_mask(slot)
assert mask.shape == (TOTAL_ACTIONS,) and mask.dtype == np.bool_ and mask.any()
print('Token空间检查通过：6个叶子、52个算子、5个窗口、142个Token')
print('token fingerprint:', action_space_fingerprint())
print('state fingerprint:', state_space_fingerprint())
print('transition fingerprint:', transition_space_fingerprint())

In [ ]:
rng = np.random.default_rng(20260805)
formulas = []
for trajectory_id in range(300):
    state = GrammarState(max_depth=4, max_nodes=12)
    while not state.done:
        slots = state.open_slots()
        slot = slots[int(rng.integers(len(slots)))]
        token_id = int(rng.choice(state.legal_token_ids(slot)))
        state = state.step(DAGAction(slot.path, token_id))

    expression = state.to_expression()
    postfix = expression.to_postfix()
    validate_postfix(postfix)
    rebuilt = Expression.from_postfix(postfix)
    assert rebuilt == expression
    assert expression.stats.node_count == state.node_count
    assert expression.stats.depth == state.max_depth_seen
    assert len(expression.structural_hash()) == 64
    assert state.count_parents() >= 1
    if trajectory_id < 5:
        formulas.append(expression.to_formula())

print('300条随机合法DAG轨迹检查通过')
for formula in formulas:
    print(' -', formula)

In [ ]:
def fill(state, name, window=0, slot_index=0):
    slot = state.open_slots()[slot_index]
    return state.step(DAGAction(slot.path, get_action_id(name, window)))

add_root = fill(GrammarState(max_depth=3, max_nodes=7), 'add')
add_left = fill(fill(add_root, 'close'), 'open')
add_right = fill(fill(add_root, 'open'), 'close')
assert add_left == add_right and add_left.count_parents() == 2

sub_root = fill(GrammarState(max_depth=3, max_nodes=7), 'sub')
sub_left = fill(fill(sub_root, 'close', slot_index=0), 'open')
sub_right_first = fill(sub_root, 'close', slot_index=1)
sub_right = fill(sub_right_first, 'open')
assert sub_left != sub_right

for operator in ('ts_corr', 'ts_cov'):
    root = fill(GrammarState(max_depth=3, max_nodes=7), operator, 5)
    left = fill(fill(root, 'close', slot_index=0), 'open')
    right_first = fill(root, 'close', slot_index=1)
    right = fill(right_first, 'open')
    assert left != right

print('阶段2验证通过：部分AST、联合mask、多路径汇合、均匀后向和保守结构去重均正常。')